# 🌦️ Notebook 03: Advanced EDA & Anomaly Detection

## Overview
This notebook covers advanced exploratory analysis:
1. Anomaly detection using Isolation Forest & LOF
2. Visualize detected anomalies on timeline
3. Seasonal decomposition analysis
4. Heatmap of weather conditions

**Input:** `data/weather_cleaned.csv`  
**Output:** Anomaly-flagged dataset + advanced visualizations

---

## Step 1: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pyod.models.iforest import IForest
from pyod.models.lof import LOF
import warnings
warnings.filterwarnings('ignore')

sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Libraries imported successfully (including PyOD)")

In [ ]:
# Load cleaned data
df = pd.read_csv("../data/weather_cleaned.csv", parse_dates=["last_updated"])

print(f"✅ Loaded dataset: {df.shape}")
print(f"   Date range: {df['last_updated'].min()} to {df['last_updated'].max()}")

## Analysis 1: Anomaly Detection (Isolation Forest)

In [ ]:
# Select features for anomaly detection
features_for_anomaly = [
    "temperature_celsius", "humidity", "wind_kph",
    "precip_mm", "pressure_mb"
]

# Prepare data (remove NaNs)
X = df[features_for_anomaly].dropna()

print(f"✅ Anomaly detection features: {len(features_for_anomaly)} variables")
print(f"   Sample size: {len(X):,} records")

# Isolation Forest
clf_if = IForest(contamination=0.02, random_state=42, n_jobs=-1)
clf_if.fit(X)
df.loc[X.index, "anomaly_iforest"] = clf_if.labels_
df.loc[X.index, "anomaly_score"] = clf_if.decision_scores_

print(f"\n✅ Isolation Forest Results:")
n_anomalies = (df["anomaly_iforest"] == 1).sum()
print(f"   Anomalies detected: {n_anomalies:,} ({n_anomalies/len(df)*100:.2f}%)")
print(f"   Normal records: {(df['anomaly_iforest'] == 0).sum():,}")

In [ ]:
# LOF (Local Outlier Factor)
clf_lof = LOF(contamination=0.02, n_neighbors=20)
clf_lof.fit(X)
df.loc[X.index, "anomaly_lof"] = clf_lof.labels_

print(f"\n✅ LOF Results:")
n_anomalies_lof = (df["anomaly_lof"] == 1).sum()
print(f"   Anomalies detected: {n_anomalies_lof:,} ({n_anomalies_lof/len(df)*100:.2f}%)")

# Compare methods
both_anomaly = ((df["anomaly_iforest"] == 1) & (df["anomaly_lof"] == 1)).sum()
print(f"   Consensus anomalies (both methods): {both_anomaly:,}")

## Analysis 2: Visualize Anomalies on Timeline

In [ ]:
# Separate normal and anomalous records
normal = df[df["anomaly_iforest"] == 0]
anomalies = df[df["anomaly_iforest"] == 1]

# Create scatter plot with anomaly highlights
fig = go.Figure()

# Normal points
fig.add_trace(go.Scatter(
    x=normal["last_updated"], 
    y=normal["temperature_celsius"],
    mode="markers", 
    marker=dict(color="steelblue", size=2, opacity=0.4),
    name="Normal"
))

# Anomalies
fig.add_trace(go.Scatter(
    x=anomalies["last_updated"], 
    y=anomalies["temperature_celsius"],
    mode="markers", 
    marker=dict(color="crimson", size=6, symbol="x"),
    name="Anomaly"
))

fig.update_layout(
    title="Temperature Anomalies Detected by Isolation Forest",
    xaxis_title="Date", 
    yaxis_title="Temperature (°C)",
    template="plotly_dark",
    height=500,
    hovermode='x unified'
)

fig.write_html("../reports/figures/06_anomaly_detection.html")
fig.show()

print(f"✅ Anomaly visualization saved")

## Analysis 3: Anomaly Characteristics

In [ ]:
# Compare statistics: normal vs anomalies
print(f"\n📊 Normal vs Anomalous Records Comparison:")
print(f"\n{'Metric':<20} {'Normal':<15} {'Anomaly':<15}")
print("="*50)

for col in features_for_anomaly:
    normal_mean = df[df["anomaly_iforest"] == 0][col].mean()
    anomaly_mean = df[df["anomaly_iforest"] == 1][col].mean()
    diff_pct = (anomaly_mean - normal_mean) / normal_mean * 100 if normal_mean != 0 else 0
    print(f"{col:<20} {normal_mean:<15.2f} {anomaly_mean:<15.2f} ({diff_pct:+.1f}%)")

In [ ]:
# Top anomalous records
print(f"\n🔴 Top 10 Most Anomalous Records (by anomaly score):")
top_anomalies = df.nlargest(10, "anomaly_score")[[
    "last_updated", "location_name", "temperature_celsius", 
    "humidity", "wind_kph", "precip_mm", "anomaly_score"
]]
print(top_anomalies.to_string(index=False))

## Analysis 4: Weather Condition Distribution

In [ ]:
# Analyze weather conditions
if "condition_text" in df.columns:
    condition_counts = df["condition_text"].value_counts().head(15)
    
    fig = px.bar(
        x=condition_counts.values,
        y=condition_counts.index,
        orientation='h',
        title="Top 15 Weather Conditions (Global)",
        color=condition_counts.values,
        color_continuous_scale="Viridis",
        template="plotly_dark",
        height=500
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    fig.write_html("../reports/figures/07_weather_conditions.html")
    fig.show()
    
    print(f"✅ Weather condition distribution:")
    print(condition_counts)
else:
    print(f"⚠️ 'condition_text' column not found in dataset")

## Analysis 5: Seasonal Pattern Decomposition

In [ ]:
# Create multi-panel seasonal analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Temperature by season
sns.boxplot(
    data=df, x="season", y="temperature_celsius",
    palette="coolwarm", ax=axes[0, 0]
)
axes[0, 0].set_title("Temperature by Season", fontweight="bold")
axes[0, 0].set_ylabel("Temperature (°C)")

# Humidity by season
sns.boxplot(
    data=df, x="season", y="humidity",
    palette="Blues", ax=axes[0, 1]
)
axes[0, 1].set_title("Humidity by Season", fontweight="bold")
axes[0, 1].set_ylabel("Humidity (%)")

# Precipitation by season
sns.boxplot(
    data=df, x="season", y="precip_mm",
    palette="RdYlGn_r", ax=axes[1, 0]
)
axes[1, 0].set_title("Precipitation by Season", fontweight="bold")
axes[1, 0].set_ylabel("Precipitation (mm)")

# Wind speed by season
sns.boxplot(
    data=df, x="season", y="wind_kph",
    palette="Greys", ax=axes[1, 1]
)
axes[1, 1].set_title("Wind Speed by Season", fontweight="bold")
axes[1, 1].set_ylabel("Wind Speed (km/h)")

plt.tight_layout()
plt.savefig("../reports/figures/08_seasonal_patterns.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"✅ Seasonal patterns visualized")

## Summary & Export

In [ ]:
# Save dataset with anomaly flags
df.to_csv("../data/weather_with_anomalies.csv", index=False)

print("\n" + "="*60)
print("ADVANCED EDA SUMMARY")
print("="*60)

print(f"\n✅ Anomaly Detection Complete:")
print(f"   • Isolation Forest: {(df['anomaly_iforest']==1).sum():,} anomalies")
print(f"   • LOF: {(df['anomaly_lof']==1).sum():,} anomalies")
print(f"   • Consensus: {both_anomaly:,} anomalies")

print(f"\n🎉 Notebook 03 Complete! Dataset with anomalies saved.")
print(f"   Ready for model building.")